<a href="https://colab.research.google.com/github/smin12345/NLP_Practice/blob/main/NLP_week04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4주차 실습: 한글 형태소 분석과 토큰화

**주제:** Kiwi·KoNLPy(Okt)를 이용한 형태소 분석, 품사 태깅, 사용자 사전

## 학습 목표
1. 어절·형태소·토큰의 차이를 설명한다.
2. 공백 기반 토큰화와 형태소 분석 기반 토큰화를 비교한다.
3. Kiwi와 Okt로 품사 태깅을 수행하고 과업에 맞는 분석기를 판단한다.
4. 사용자 사전이 필요한 사례를 만들고 적용한다.

> **제출:** 모든 TODO를 작성·실행하고, 체크포인트 답변을 Markdown 셀에 남기세요. 외부 AI·자료를 사용했다면 사용 범위와 검증 과정을 기록하세요.

## 0. 준비
교육용 예시 데이터입니다. 실제 데이터는 출처·라이선스·개인정보 포함 여부를 확인하세요.

In [2]:
# 실행 오류가 날 때만 아래 주석을 해제하세요.
!pip -q install -U kiwipiepy konlpy

import re
import pandas as pd
from kiwipiepy import Kiwi
from konlpy.tag import Okt

kiwi = Kiwi()
okt = Okt()

texts = [
    '충북대학교 자연어처리 수업이 정말 재미있어요!',
    'AI 챗봇의 답변 품질을 개선하고 싶어요.',
    '배터리가 오래가고 화면도 선명해서 만족합니다.',
    '결말은 아쉬웠지만 배우 연기는 최고였어요.',
    '새로 나온 갓생 챌린지 앱을 사용해 봤어요.'
]
df = pd.DataFrame({'text_id': range(1, len(texts)+1), 'text': texts})
display(df)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 11.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 27.5 MB/s eta 0:00:00


,text_id,text
0,1,충북대학교 자연어처리 수업이 정말 재미있어요!
1,2,AI 챗봇의 답변 품질을 개선하고 싶어요.
2,3,배터리가 오래가고 화면도 선명해서 만족합니다.
3,4,결말은 아쉬웠지만 배우 연기는 최고였어요.
4,5,새로 나온 갓생 챌린지 앱을 사용해 봤어요.


## 1. 어절·형태소·토큰
- **어절:** 띄어쓰기로 구분되는 단위
- **형태소:** 뜻 또는 문법 기능을 지닌 최소 단위
- **토큰:** 분석 목적에 맞게 모델 입력으로 정한 단위

예: `학생들이 도서관에서 공부한다.` → `학생/들/이`, `도서관/에서`, `공부/하/ㄴ다`처럼 분해할 수 있습니다.

In [3]:
sample = '학생들이 도서관에서 공부한다.'
print('원문:', sample)
print('공백 토큰:', sample.split())
print('정규표현식 토큰:', re.findall(r'[가-힣A-Za-z0-9]+', sample))

# TODO 1: Kiwi 형태소 분석 결과를 [(형태, 품사), ...] 형태로 출력하세요.
# kiwi_result = ...
# print(kiwi_result)
kiwi_result = [(token.form, token.tag) for token in kiwi.tokenize(sample)]
print(kiwi_result)

원문: 학생들이 도서관에서 공부한다.
공백 토큰: ['학생들이', '도서관에서', '공부한다.']
정규표현식 토큰: ['학생들이', '도서관에서', '공부한다']
[('학생', 'NNG'), ('들', 'XSN'), ('이', 'JKS'), ('도서관', 'NNG'), ('에서', 'JKB'), ('공부', 'NNG'), ('하', 'XSV'), ('ᆫ다', 'EF'), ('.', 'SF')]


### 체크포인트 1
공백 토큰화로는 보이지 않는 문법·의미 정보 한 가지를 위 문장에서 찾아 1~2문장으로 설명하세요.

공백 토큰화에서는 ‘학생들이’가 하나의 단어로 보이지만, 형태소 분석을 하면 ‘학생 + 들 + 이’로 나뉜다. 이를 통해 ‘들’은 복수, ‘이’는 주격 조사라는 문법 정보를 알 수 있다.

## 2. Kiwi로 형태소 분석과 품사 태깅
Kiwi는 현대 한국어 분석에 활용하기 좋은 형태소 분석기입니다. `token.form`은 형태, `token.tag`는 품사 태그입니다.

In [4]:
def kiwi_pos(text):
    return [(token.form, token.tag) for token in kiwi.tokenize(text)]

for text in df['text']:
    print(text)
    print(kiwi_pos(text))
    print()

# TODO 2: 아래 문장의 명사(태그가 N으로 시작)만 추출하는 함수를 작성하세요.
# def kiwi_nouns(text):
    # return [...]
    # pass
def kiwi_nouns(text):
    return [token.form for token in kiwi.tokenize(text) if token.tag.startswith('N')]

print(kiwi_nouns(df.loc[0, 'text']))



# print(kiwi_nouns(df.loc[0, 'text']))

충북대학교 자연어처리 수업이 정말 재미있어요!
[('충북대학교', 'NNP'), ('자연어 처리', 'NNP'), ('수업', 'NNG'), ('이', 'JKS'), ('정말', 'MAG'), ('재미있', 'VA'), ('어요', 'EF'), ('!', 'SF')]

AI 챗봇의 답변 품질을 개선하고 싶어요.
[('AI', 'SL'), ('챗봇', 'NNG'), ('의', 'JKG'), ('답변', 'NNG'), ('품질', 'NNG'), ('을', 'JKO'), ('개선', 'NNG'), ('하', 'XSV'), ('고', 'EC'), ('싶', 'VX'), ('어요', 'EF'), ('.', 'SF')]

배터리가 오래가고 화면도 선명해서 만족합니다.
[('배터리', 'NNG'), ('가', 'JKS'), ('오래', 'MAG'), ('가', 'VV'), ('고', 'EC'), ('화면', 'NNG'), ('도', 'JX'), ('선명', 'NNG'), ('하', 'XSA'), ('어서', 'EC'), ('만족', 'NNG'), ('하', 'XSA'), ('ᆸ니다', 'EF'), ('.', 'SF')]

결말은 아쉬웠지만 배우 연기는 최고였어요.
[('결말', 'NNG'), ('은', 'JX'), ('아쉽', 'VA-I'), ('었', 'EP'), ('지만', 'EC'), ('배우', 'NNG'), ('연기', 'NNG'), ('는', 'JX'), ('최고', 'NNG'), ('이', 'VCP'), ('었', 'EP'), ('어요', 'EF'), ('.', 'SF')]

새로 나온 갓생 챌린지 앱을 사용해 봤어요.
[('새로', 'MAG'), ('나오', 'VV'), ('ᆫ', 'ETM'), ('갓', 'MAG'), ('생', 'XPN'), ('챌린지', 'NNG'), ('앱', 'NNG'), ('을', 'JKO'), ('사용', 'NNG'), ('하', 'XSV'), ('어', 'EC'), ('보', 'VX'), ('었', 'EP'), ('어요', 'E

## 3. KoNLPy의 Okt로 형태소 분석
Okt는 소셜 텍스트·이모티콘·비표준 표현을 살펴볼 때 자주 사용합니다. 분석기마다 분리 방식과 품사 체계가 다를 수 있습니다.

In [5]:
compare_text = '오늘 수업 진짜 재밌었는데 과제는 조금 어렵네요 ㅠㅠ'
print('원문:', compare_text)
print('Kiwi:', kiwi_pos(compare_text))
print('Okt :', okt.pos(compare_text))

# TODO 3: 아래 문장을 Kiwi와 Okt로 각각 분석하고 결과 차이를 관찰하세요.
# my_sentence = '신조어와 줄임말도 분석할 수 있을까?'
# print('Kiwi:', ...)
# print('Okt :', ...)
my_sentence = '신조어와 줄임말도 분석할 수 있을까?'

print('Kiwi:', kiwi_pos(my_sentence))
print('Okt :', okt.pos(my_sentence))

원문: 오늘 수업 진짜 재밌었는데 과제는 조금 어렵네요 ㅠㅠ
Kiwi: [('오늘', 'NNG'), ('수업', 'NNG'), ('진짜', 'MAG'), ('재밌', 'VA'), ('었', 'EP'), ('는데', 'EC'), ('과제', 'NNG'), ('는', 'JX'), ('조금', 'MAG'), ('어렵', 'VA-I'), ('네요', 'EF'), ('ㅠㅠ', 'SW')]
Okt : [('오늘', 'Noun'), ('수업', 'Noun'), ('진짜', 'Noun'), ('재밌었는데', 'Adjective'), ('과제', 'Noun'), ('는', 'Josa'), ('조금', 'Noun'), ('어렵네요', 'Adjective'), ('ㅠㅠ', 'KoreanParticle')]
Kiwi: [('신조어', 'NNG'), ('와', 'JC'), ('줄임', 'NNG'), ('말', 'NNG'), ('도', 'JX'), ('분석', 'NNG'), ('하', 'XSV'), ('ᆯ', 'ETM'), ('수', 'NNB'), ('있', 'VA'), ('을까', 'EF'), ('?', 'SF')]
Okt : [('신조어', 'Noun'), ('와', 'Josa'), ('줄임말도', 'Verb'), ('분석', 'Noun'), ('할', 'Verb'), ('수', 'Noun'), ('있을까', 'Adjective'), ('?', 'Punctuation')]


### 체크포인트 2
Kiwi와 Okt 결과에서 토큰 분리 또는 품사 태깅이 다른 사례를 하나 기록하세요. 어느 분석기가 “더 좋다”가 아니라, 어떤 **데이터/과업**에서 더 편리할지 설명하세요.


'재밌었는데’를 Kiwi는 ‘재밌/었/는데’로 세분화한 반면, Okt는 ‘재밌었는데’를 하나의 형용사로 분석했다. 따라서 어미나 시제처럼 세부적인 문법 정보를 분석할 때는 Kiwi가 편리하고, 리뷰나 SNS처럼 전체적인 단어와 감성을 간단히 분석할 때는 Okt가 편리할 수 있다.

## 4. 과업에 따른 품사 선택
키워드 추출은 명사 중심 처리가 유용할 수 있습니다. 반면 감성 분석에서는 형용사·동사·부정 표현·강조 부사도 중요한 신호가 됩니다.

In [6]:

review = '배송은 조금 느렸지만 제품 디자인이 매우 예쁘고 만족스러워요.'
print('전체 분석:', kiwi_pos(review))

# TODO 4: 명사(N*)와 용언(V*)을 각각 추출하세요.
def select_pos(text):
    nouns = []
    predicates = []

    for token in kiwi.tokenize(text):
        if token.tag.startswith('N'):
            nouns.append(token.form)
        elif token.tag.startswith('V'):
            predicates.append(token.form)

    return nouns, predicates

nouns, predicates = select_pos(review)
print('명사:', nouns)
print('용언:', predicates)

전체 분석: [('배송', 'NNG'), ('은', 'JX'), ('조금', 'MAG'), ('느리', 'VA'), ('었', 'EP'), ('지만', 'EC'), ('제품', 'NNG'), ('디자인', 'NNG'), ('이', 'JKS'), ('매우', 'MAG'), ('예쁘', 'VA'), ('고', 'EC'), ('만족', 'NNG'), ('스럽', 'XSA-I'), ('어요', 'EF'), ('.', 'SF')]
명사: ['배송', '제품', '디자인', '만족']
용언: ['느리', '예쁘']


### 체크포인트 3
`좋지 않아요`, `매우 예쁘고` 같은 표현을 생각해 보세요. 명사만 남기면 감성 분석에서 어떤 정보가 사라질까요?

명사만 남기면 문장의 긍정·부정 여부와 감정의 정도를 정확하게 파악하기 어려워질 수 있다.

## 5. 사용자 사전: 왜 필요한가
새 서비스명, 과목명, 브랜드명, 전문 용어는 일반 사전에 없거나 원하지 않는 방식으로 분리될 수 있습니다. 사용자 사전은 **도메인 용어를 원하는 품사와 단위로 분석**하도록 돕습니다.

아래 코드는 Kiwi 버전에 따라 사용자 단어 추가 방식이 다를 수 있으므로, 실행 환경의 오류 메시지와 라이브러리 문서를 함께 확인하세요.

In [12]:
domain_text = '충북대학교에서 자연어처리, 머신 러닝, 딥러닝 프로젝트를 진행한다.'
print('추가 전:', kiwi_pos(domain_text))

# TODO 5: 사용자 단어 추가
kiwi.add_user_word('머신 러닝', 'NNP')
kiwi.add_user_word('딥러닝', 'NNP')

print('추가 후:', kiwi_pos(domain_text))

# TODO 6: 전공/관심 분야 고유명사 2개 추가
my_text = '컴퓨터공학에서 인공지능을 공부한다.'

print('사용자 단어 추가 전:', kiwi_pos(my_text))

kiwi.add_user_word('컴퓨터공학', 'NNP')
kiwi.add_user_word('인공지능', 'NNP')

print('사용자 단어 추가 후:', kiwi_pos(my_text))

추가 전: [('충북대학교', 'NNP'), ('에서', 'JKB'), ('자연어 처리', 'NNP'), (',', 'SP'), ('머신 러닝', 'NNP'), (',', 'SP'), ('딥러닝', 'NNP'), ('프로젝트', 'NNG'), ('를', 'JKO'), ('진행', 'NNG'), ('하', 'XSV'), ('ᆫ다', 'EF'), ('.', 'SF')]
추가 후: [('충북대학교', 'NNP'), ('에서', 'JKB'), ('자연어 처리', 'NNP'), (',', 'SP'), ('머신 러닝', 'NNP'), (',', 'SP'), ('딥러닝', 'NNP'), ('프로젝트', 'NNG'), ('를', 'JKO'), ('진행', 'NNG'), ('하', 'XSV'), ('ᆫ다', 'EF'), ('.', 'SF')]
사용자 단어 추가 전: [('컴퓨터공학', 'NNP'), ('에서', 'JKB'), ('인공지능', 'NNP'), ('을', 'JKO'), ('공부', 'NNG'), ('하', 'XSV'), ('ᆫ다', 'EF'), ('.', 'SF')]
사용자 단어 추가 후: [('컴퓨터공학', 'NNP'), ('에서', 'JKB'), ('인공지능', 'NNP'), ('을', 'JKO'), ('공부', 'NNG'), ('하', 'XSV'), ('ᆫ다', 'EF'), ('.', 'SF')]


## 6. 미니 과제: 토큰화 정책 설계
다음 목표 중 하나를 선택해 토큰화 정책을 설계하세요.
1. 강의평 키워드 추출
2. 영화 리뷰 감성 분석
3. 학교 민원 자동 분류

아래 함수에 분석기, 보존할 품사, 제거할 토큰의 기준을 작성하세요.

In [15]:
def tokenize_for_task(text):
    """TODO 7: 선택한 과업에 맞는 토큰화 규칙을 구현하세요.
    - 어떤 품사를 남길지
    - 조사/기호/URL 등을 어떻게 다룰지
    - 부정 표현·이모지·반복 문자를 보존할지
    """
    tokens = []
    # TODO
    for token in kiwi.tokenize(text):
        # 명사, 동사, 형용사, 부사
        if token.tag.startswith(('N', 'V')) or token.tag == 'MAG':
            tokens.append(token.form)

        # 이모티콘, 특수 표현
        elif token.tag == 'SW':
            tokens.append(token.form)

    return tokens

for text in df['text']:
    print(text, '->', tokenize_for_task(text))


# 샘플 문장 3개
samples = [
    '영화가 정말 재미있어요 ㅋㅋㅋ',
    '스토리가 별로 좋지 않아요 ㅠㅠ',
    '배우 연기는 매우 좋았어요!'
]

for text in samples:
    print(text, '->', tokenize_for_task(text))

충북대학교 자연어처리 수업이 정말 재미있어요! -> ['충북대학교', '자연어 처리', '수업', '정말', '재미있']
AI 챗봇의 답변 품질을 개선하고 싶어요. -> ['챗봇', '답변', '품질', '개선', '싶']
배터리가 오래가고 화면도 선명해서 만족합니다. -> ['배터리', '오래', '가', '화면', '선명', '만족']
결말은 아쉬웠지만 배우 연기는 최고였어요. -> ['결말', '아쉽', '배우', '연기', '최고', '이']
새로 나온 갓생 챌린지 앱을 사용해 봤어요. -> ['새로', '나오', '갓', '챌린지', '앱', '사용', '보']
영화가 정말 재미있어요 ㅋㅋㅋ -> ['영화', '정말', '재미있', 'ㅋㅋㅋ']
스토리가 별로 좋지 않아요 ㅠㅠ -> ['스토리', '별로', '좋', '않', 'ㅠㅠ']
배우 연기는 매우 좋았어요! -> ['배우', '연기', '매우', '좋']


영화 리뷰 감성 분석을 진행하였다. 감성 판단에 필요한다고 생각되는 명사, 동사, 형용사, 부사를 남기고 조사는 제거하였다. 또한 ‘ㅠㅠ’, ‘ㅋㅋㅋ’ 같은 표현도 감정을 나타낼 수 있기 때문에 보존하도록 하였다. ‘않다’와 같은 부정 표현도 문장의 감성을 바꿀 수 있으므로 분석할 때 중요한 정보로 고려하였다.

### 체크포인트 4
선택한 과업, 분석기 선택 이유, 보존/제거 규칙, 예상되는 오류 사례를 각각 1문장 이상 작성하세요.

영화 리뷰의 감성 분석을 진행하였다.
분석기는 형용사와 어미 등을 세부적으로 구분할 수 있는 Kiwi를 선택하였다.
감성 판단에 필요한 명사, 동사, 형용사, 부사와 ‘ㅋㅋㅋ’, ‘ㅠㅠ’ 같은 감정 표현은 보존하고, 조사와 일반적인 문장 기호는 제거하였다.
예상되는 오류로는 신조어나 줄임말을 잘못 분리하거나, ‘좋지 않다’와 같은 부정 표현을 정확하게 처리하지 못해 실제 감성과 다르게 판단하는 경우가 있을 수 있다고 생각한다.

퀴즈 1: 교착어
퀴즈 2: 사용자 사전

## 제출 체크리스트
- [ ] TODO 1~7을 작성하고 실행했다.
- [ ] 공백·정규표현식·형태소 기반 토큰화를 비교했다.
- [ ] Kiwi와 Okt의 결과 차이 및 과업 적합성을 설명했다.
- [ ] 명사 중심 처리의 장점과 한계를 설명했다.
- [ ] 사용자 사전에 고유명사 2개 이상을 추가하고 전후 결과를 비교했다.
- [ ] 선택 과업에 대한 토큰화 정책과 오류 사례를 기록했다.
- [ ] 외부 AI/자료 사용 시 범위와 검증 과정을 기록했다.

## 다음 주차 예고
전처리된 토큰을 Bag-of-Words, TF-IDF, n-gram으로 수치화하여 문서 비교와 분류 모델의 입력 특징을 구성합니다.